In [138]:
import pandas as pd

from Roadmap.DataEngineering.pipelines import features

data = pd.read_csv('data/4k/bank.csv', sep=';')

In [139]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        4521 non-null   int64 
 1   job        4521 non-null   object
 2   marital    4521 non-null   object
 3   education  4521 non-null   object
 4   default    4521 non-null   object
 5   balance    4521 non-null   int64 
 6   housing    4521 non-null   object
 7   loan       4521 non-null   object
 8   contact    4521 non-null   object
 9   day        4521 non-null   int64 
 10  month      4521 non-null   object
 11  duration   4521 non-null   int64 
 12  campaign   4521 non-null   int64 
 13  pdays      4521 non-null   int64 
 14  previous   4521 non-null   int64 
 15  poutcome   4521 non-null   object
 16  y          4521 non-null   object
dtypes: int64(7), object(10)
memory usage: 600.6+ KB


In [140]:
num_features = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous', 'day']
cat_features = ['job', 'marital', 'education', 'housing', 'loan', 'contact', 'month', 'poutcome']

In [141]:
X  = data[num_features + cat_features]
y = data['default']
y = y.map({'no': 0, 'yes': 1})

In [142]:
cnt = (y == 1).sum()

In [143]:
cnt

np.int64(76)

In [144]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [145]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    auto_class_weights='Balanced',
    random_seed=42,
    early_stopping_rounds=50,
    eval_metric='PRAUC'
)

In [146]:
model.fit(X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_features,
    verbose=100)

0:	learn: 0.7794259	test: 0.5428825	best: 0.5428825 (0)	total: 70.6ms	remaining: 1m 10s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9086395057
bestIteration = 19

Shrink model to first 20 iterations.


In [147]:
import numpy as np
from sklearn.metrics import precision_recall_curve, f1_score
from sklearn.metrics import recall_score, precision_score

y_proba = model.predict_proba(X_val)[:, 1]

best_thresh = 0.5
best_prec = 0.0
best_rec = 0.0
best_f1 = 0.0

for thresh in np.linspace(0.05, 0.95, 91):
    y_pred_temp = (y_proba >= thresh).astype(int)
    rec = recall_score(y_val, y_pred_temp, zero_division=0)
    prec = precision_score(y_val, y_pred_temp, zero_division=0)
    f1 = f1_score(y_val, y_pred_temp)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh
        best_rec = rec
        best_prec = prec


print(f"Оптимальный порог: {best_thresh:.3f}")
print(f"Recall: {best_rec * 100:.2f}%")
print(f"Precision: {best_prec * 100:.2f}%")
print(f"F1-score: {best_f1 * 100:.2f}%")

Оптимальный порог: 0.650
Recall: 40.00%
Precision: 26.09%
F1-score: 31.58%
